# Web IQ: REST API 직접 호출

## 목표와 사전 준비

포털 밖의 Python에서 Microsoft Foundry와 Microsoft Copilot을 Web, Images, Videos로 검색합니다. **고객 실습에 사용이 허용된 진행자 제공 키로 참가자가 직접 실행**합니다. 포털 접근·키 발급은 참가자에게 전제하지 않으며, 키가 없으면 저장된 실제 출력과 [설명 자료](README.md)를 분석합니다.

- Python 3.11 이상, VS Code Jupyter 확장, 인터넷 연결이 필요합니다.
- 진행자가 [Profile Management](https://webiq.microsoft.ai/profiles)에서 발급해 제공한 실습 키를 `WEBIQ_API_KEY` 환경 변수 또는 숨김 입력으로 전달합니다. 키를 코드에 붙여 넣지 않습니다.
- 실행 위치는 이 노트북입니다. 위에서 아래로 실행하며, 검색 셀은 6회 API 호출을 사용합니다. 계정별 허용 서비스·할당량·요금을 먼저 확인하세요.
- REST는 `POST https://api.microsoft.ai/v3/search/{vertical}`와 `x-apikey` 헤더를 사용합니다. LLM 호출이나 Azure 리소스 배포는 필요하지 않습니다.

공식 문서: [Web](https://webiq.microsoft.ai/documentation/api-reference/web/) · [Images](https://webiq.microsoft.ai/documentation/api-reference/images/) · [Videos](https://webiq.microsoft.ai/documentation/api-reference/videos/) · [인증](https://webiq.microsoft.ai/documentation/authentication/)

In [1]:
%pip install "httpx>=0.28,<1"

Looking in indexes: https://packagefeedproxy.microsoft.io/pypi/simple/



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: /Users/changjuahn/Repo/MicrosoftIQSeries-KR/.venv/bin/python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
import getpass
import json
import os
from datetime import datetime, timezone

import httpx

api_key = (os.environ.get("WEBIQ_API_KEY") or getpass.getpass("Web IQ API key: ")).strip()
if not api_key:
    raise ValueError("Web IQ API 키가 필요합니다.")
print("API 키 준비 완료. 키 값은 출력하지 않습니다.")
print("실행 시각 (UTC):", datetime.now(timezone.utc).isoformat())

API 키 준비 완료. 키 값은 출력하지 않습니다.
실행 시각 (UTC): 2026-09-30T01:33:58.922189+00:00


In [3]:
results = {}
result_fields = {"web": "webResults", "images": "imageResults", "videos": "videoResults"}

with httpx.Client(
    base_url="https://api.microsoft.ai",
    headers={"x-apikey": api_key},
    timeout=60.0,
) as client:
    for topic in ("Microsoft Foundry", "Microsoft Copilot"):
        results[topic] = {}
        for vertical, result_field in result_fields.items():
            query = f"{topic} YouTube" if vertical == "videos" else topic
            payload = {"query": query, "maxResults": 3, "language": "en", "region": "US"}
            if vertical == "web":
                payload.update(contentFormat="passage", maxLength=800)

            endpoint = f"/v3/search/{vertical}"
            response = client.post(endpoint, json=payload)
            print(f"\nPOST https://api.microsoft.ai{endpoint} | {query} | HTTP {response.status_code}")
            if response.status_code in (401, 403, 429):
                reasons = {401: "키가 유효한지 확인하세요.", 403: "계정의 서비스 권한을 확인하세요.", 429: "할당량과 호출 빈도를 확인하세요."}
                raise RuntimeError(f"HTTP {response.status_code}: {reasons[response.status_code]}")
            response.raise_for_status()
            result = response.json()
            items = result.get(result_field, [])
            if not items:
                raise RuntimeError(f"{topic} / {vertical}: 반환 결과가 없습니다. 검색어와 응답을 확인하세요.")
            results[topic][vertical] = result
            print(f"{result_field}: {len(items)}건 | traceId: {result.get('traceId', '(없음)')}")
            for item in items:
                print("제목:", item.get("title", "(없음)"))
                print("URL:", item.get("url", "(없음)"))
                print("내용:", (item.get("content") or item.get("caption") or item.get("description") or "")[:220])
                if vertical == "images":
                    print("원문 페이지:", item.get("hostPageUrl", "(없음)"))
                if vertical == "videos":
                    print("게시자:", item.get("publishedBy", "(없음)"))


POST https://api.microsoft.ai/v3/search/web | Microsoft Foundry | HTTP 200
webResults: 3건 | traceId: 6abc6707c1f84e318dfb42d42cda851f
제목: Microsoft Foundry | Microsoft Azure
URL: https://azure.microsoft.com/en-us/products/ai-foundry/
내용: Microsoft Foundry | Microsoft Azure Microsoft Foundry Microsoft Foundry is a unified platform to build, ground, and govern AI apps and agents that understand your business context. It brings together the full agent lifec
제목: What is Microsoft Foundry? - Microsoft Foundry | Microsoft Learn
URL: https://learn.microsoft.com/en-us/azure/foundry/what-is-foundry
내용: What is Microsoft Foundry? - Microsoft Foundry | Microsoft Learn Learn Azure Microsoft Foundry Ask Learn What is Microsoft Foundry? Summarize this article for me In this article Get started What you can build Enterprise-
제목: Microsoft Foundry documentation | Microsoft Learn
URL: https://learn.microsoft.com/en-us/azure/foundry/
내용: Microsoft Foundry documentation | Microsoft Learn Microsoft Found


POST https://api.microsoft.ai/v3/search/images | Microsoft Foundry | HTTP 200
imageResults: 3건 | traceId: 6abc6708370246c3b3ecce4f2477dfdb
제목: Microsoft Foundry: When a Rebrand Signals Something Deeper
URL: https://thepowerplatformcave.com/content/images/2025/11/Gemini_Generated_Image_alngzralngzralng.png
내용: A photo of a large industrial furnace with bright orange and yellow flames. The Microsoft logo is prominently displayed in the center, with 'Microsoft Foundry' written below it. The background shows a complex structure o
원문 페이지: https://thepowerplatformcave.com/microsoft-foundry-ignite-2025-analysis/
제목: Microsoft Mechanics Podcast: Introducing Copilot in the Micr...
URL: https://static.libsyn.com/p/assets/2/f/0/5/2f050d0bb423d0bed959afa2a1bf1c87/Foundry_Podcast.jpg
내용: A product photo of Microsoft Foundry, an AI app and agent factory. The interface shows a central node with connected nodes and a flowchart. Icons on the left represent modules and tools. The background is light pu


POST https://api.microsoft.ai/v3/search/videos | Microsoft Foundry YouTube | HTTP 200
videoResults: 3건 | traceId: 6abc6708bdf44db9a00110b4b66a76f9
제목: Build, test, monitor and optimize a voice agent in Microsoft Foundry
URL: https://www.youtube.com/watch?v=u8usaIQMR5s
내용: An in-depth walkthrough of building a voice agent in Microsoft Foundry—from first configuration through to production monitoring—using a real estate agent scenario. Our product lead shows how to: • Create and configure a
게시자: Microsoft Azure
제목: Microsoft Foundry Full Tutorial: From Basics to Agentic AI (2026
URL: https://www.youtube.com/watch?v=XHealU2WXho
내용: ⚡ WANT ACCESS TO 100s OF MICROSOFT AI & POWER PLATFORM CHEAT SHEETS, ROADMAPS, AND PLAYBOOKS?? Get your FREE Success Kit here: https://www.powercademy.com/success-kit — Microsoft Foundry is Microsoft's unified platform f
게시자: Powercademy
제목: Streamlining Model Onboarding and Exploration in Microsoft Foundry
URL: https://www.youtube.com/watch?v=eCS00D0UHdo
내용: 


POST https://api.microsoft.ai/v3/search/web | Microsoft Copilot | HTTP 200
webResults: 3건 | traceId: 6abc67085aaf4bf0ad319168c50fa200
제목: Microsoft Copilot: AI built for work
URL: https://www.microsoft.com/en-us/copilot
내용: Microsoft Copilot: AI built for work Copilot connects with your work content and the apps you already use, bringing together the latest AI models to help you create finished work, not just answers. Copilot connects with 
제목: What Is a Copilot and How Does It Work? | Microsoft Copilot
URL: https://www.microsoft.com/en/microsoft-copilot/copilot-101/what-is-copilot
내용: What Is a Copilot and How Does It Work? | Microsoft Copilot Microsoft Copilot Copilot 101 What is Copilot What is a copilot? Discover what a copilot is, how to use a copilot, and how a copilot can boost productivity. Exp
제목: Microsoft 365 Copilot | AI Productivity Tools for Work
URL: https://www.microsoft.com/en-ie/microsoft-365-copilot/
내용: Microsoft 365 Copilot | AI Productivity Tools for Work Read th


POST https://api.microsoft.ai/v3/search/images | Microsoft Copilot | HTTP 200
imageResults: 3건 | traceId: 6abc6709c3b14747b5168c6ddd8ca750
제목: 7 Things to Know Before Implementing Microsoft Copilot Company-Wide - TagVA
URL: https://www.tagva.com/wp-content/uploads/2025/05/Microsoft-Copilot-scaled.jpg
내용: A photo of a laptop screen displaying the Microsoft Copilot interface. The browser tab shows 'copilot.mstelevr.com' and the URL is visible. The screen features the Copilot logo, a colorful square with three colored squar
원문 페이지: https://www.tagva.com/7-things-to-know-before-implementing-microsoft-copilot-company-wide/
제목: Microsoft Copilot | IT.ie
URL: https://it.ie/wp-content/uploads/2026/04/Microsogy-Copilot-Hero.png
내용: A product photo of a colorful, abstract design with a ribbon-like structure. The ribbon features a gradient of purple, pink, and blue colors. On the left side, there are icons for Word, Excel, and PowerPoint. A notificat
원문 페이지: https://it.ie/microsoft-copilot/
제목: 


POST https://api.microsoft.ai/v3/search/videos | Microsoft Copilot YouTube | HTTP 200
videoResults: 3건 | traceId: 6abc670990174379bb9eac6e25302f63
제목: The New Microsoft Copilot Is Here: Everything You Need to Know
URL: https://www.youtube.com/watch?v=bar6CRErgC4
내용: *Try Microsoft Copilot* - https://toolfinder.com/go/co-pilot The new Microsoft Copilot is split into three areas: Home, Code and Autopilot. This explainer covers what each one does, from Chat and Cowork working with Word
게시자: Tool Finder
제목: The new Copilot: where AI-powered work comes together
URL: https://www.youtube.com/watch?v=OgInADh5Tcs
내용: Meet the new Microsoft Copilot: one experience where AI-powered work comes together. Home is your new starting point, where Chat and Cowork come together with the full power of Office. Code empowers anyone to build solut
게시자: Microsoft Copilot
제목: Microsoft Copilot Just Got a Massive Upgrade
URL: https://www.youtube.com/watch?v=KJ8Y2Nb9y9o
내용: Microsoft just unveiled a completely 

## 결과 해석과 정리

- `webResults`: 제목, 출처 URL, `content`를 확인합니다. `passage`는 검색어에 관련된 문맥을 추출합니다. REST Web에는 `snippet` 필드가 없습니다.
- `imageResults`: 이미지 URL과 `hostPageUrl`은 다릅니다. `caption`은 생성된 설명이며 이미지의 사용 권리를 보장하지 않습니다.
- `videoResults`: `publishedBy`, `url`, `length`를 확인합니다. `YouTube`를 검색어에 넣어도 YouTube 전용 필터가 되는 것은 아닙니다. 실제 반환 URL을 확인합니다. `summary`, `moments` 등은 없을 수 있습니다.
- 검색 순위·본문·반환 건수는 실행 시점과 권한에 따라 달라집니다. 검색 성공이 원문의 정확성이나 접근 가능성을 보장하지 않습니다.
- 401은 키, 403은 서비스 권한, 429는 할당량을 확인합니다. 그 외 HTTP 오류는 상태 코드로 확인하며 자동 재시도하지 않습니다.

아래 셀은 첫 웹 결과의 일부를 원래 필드명으로 출력합니다. 전체 원문 대신 짧은 발췌만 공유합니다. 이후 MCP 실습까지 마치면 **이번에 만든 데모 키만** Profile Management에서 폐기하세요. 기존 키는 삭제하지 않습니다. 커널을 재시작해 메모리의 키·클라이언트·응답도 정리합니다.

In [4]:
first_result = results["Microsoft Foundry"]["web"]["webResults"][0]
print(json.dumps({
    "title": first_result.get("title"),
    "url": first_result.get("url"),
    "content_excerpt": first_result.get("content", "")[:220],
}, ensure_ascii=False, indent=2))
del api_key
print("키 변수를 제거했습니다. 실습 종료 후 커널을 재시작하고 데모 전용 키를 폐기하세요.")

{
  "title": "Microsoft Foundry | Microsoft Azure",
  "url": "https://azure.microsoft.com/en-us/products/ai-foundry/",
  "content_excerpt": "Microsoft Foundry | Microsoft Azure Microsoft Foundry Microsoft Foundry is a unified platform to build, ground, and govern AI apps and agents that understand your business context. It brings together the full agent lifec"
}
키 변수를 제거했습니다. 실습 종료 후 커널을 재시작하고 데모 전용 키를 폐기하세요.
